# BERTSubs quickstart: call BERTSubs directly through DeepOnto

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HuiYang1997/OnT/blob/main/notebooks/BERTSubs_Quickstart.ipynb)

**BERTSubs** (Chen et al., WWW Journal 2023) fine-tunes BERT on an ontology's own subclass axioms, then predicts the probability of new ones.
Three core steps with [DeepOnto](https://krr-oxford.github.io/DeepOnto/): **set up**, **run**, **read the results**.

Use a **GPU** runtime (*Runtime → Change runtime type → T4 GPU*), then **Runtime → Run all**.

## Setup · run once

On Linux: `sudo apt install openjdk-17-jre` and `pip install deeponto==0.9.3 transformers==4.45.2 "datasets<4"`.

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules:                     # Java for the OWL API
    subprocess.run("apt-get -qq install -y openjdk-17-jre-headless", shell=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
%pip install -q deeponto==0.9.3 transformers==4.45.2 "datasets<4"

import random                                         # for Python 3.11+
_sample = random.sample
random.sample = lambda s, k, **kw: _sample(sorted(s, key=str) if isinstance(s, set) else s, k, **kw)
from deeponto import init_jvm
init_jvm("4g")
import deeponto.complete.bertsubs.pipeline_intra as bp   # small ontology: keep the final model
_TA = bp.TrainingArguments
bp.TrainingArguments = lambda **kw: _TA(**{**kw, "load_best_model_at_end": False})

INFO:deeponto:4g maximum memory allocated to JVM.


INFO:deeponto:JVM started successfully.


## The ontology

`family_tree.owl`: 31 classes and 42 subclass axioms. `aunt` and `niece` are declared but have no parents yet: BERTSubs will find them.

In [2]:
import urllib.request
if not os.path.exists("family_tree.owl"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HuiYang1997/OnT/main/notebooks/family_tree.owl", "family_tree.owl")

In [3]:
from deeponto.onto import Ontology
onto = Ontology("family_tree.owl", "hermit")       # downloaded from the notebook folder
NS = "http://example.org/family#"
father = onto.get_owl_object(NS + "Father")
print(len(onto.owl_classes), "classes")
print(onto.get_annotations(father, "http://www.w3.org/2000/01/rdf-schema#label"))
print([iri[len(NS):] for iri in onto.reasoner.get_inferred_super_entities(father, direct=False)])

31 classes
['father']
['Male', 'Person', 'Parent']


## Core step 1 · Set up BERTSubs

The axioms to check (a class, then its possible parents), the training pairs (every subclass axiom of the ontology), and DeepOnto's default settings.

In [4]:
with open("candidates.csv", "w") as f:            # a class, then its possible parents
    for c in ["Aunt", "Niece"]:
        f.write(",".join(NS + x for x in [c, "Female", "Male"]) + "\n")

In [5]:
from deeponto.complete.bertsubs import BERTSubsIntraPipeline, DEFAULT_CONFIG_FILE_INTRA
pairs = BERTSubsIntraPipeline.extract_subsumptions_from_ontology(onto, "named_class")
with open("train.csv", "w") as f:                 # every stated C ⊑ D
    for ax in pairs:
        f.write(f"{ax.getSubClass().getIRI()},{ax.getSuperClass().getIRI()}\n")

In [6]:
from yacs.config import CfgNode
from deeponto.utils import load_file
config = CfgNode(load_file(DEFAULT_CONFIG_FILE_INTRA))
config.onto_file = "family_tree.owl"
config.train_subsumption_file = config.valid_subsumption_file = "train.csv"
config.test_subsumption_file = "candidates.csv"
config.test_type = "prediction"                 # write P(C ⊑ D) for every candidate
config.fine_tune.pretrained = "bert-base-uncased"
config.fine_tune.num_epochs = 10
config.fine_tune.batch_size = 16
config.fine_tune.train_neg_dup = 16              # random negatives per positive

## Core step 2 · Run

One call trains BERT (it reads both class names together, as one input) and then checks each axiom.

In [7]:
pipeline = BERTSubsIntraPipeline(onto=onto, config=config)

31 named classes, 1.0 labels per class
Positive train/valid subsumptions: 42/42


fine-tuning done, fine-tuned model saved
Fine-tuning costs 4.6 minutes
Predicted subsumption scores are saved to test_subsumption_scores.csv


## Core step 3 · Read the results

In [8]:
import pandas as pd
def read_scores():                   # pair each axiom with its probability
    rows = [l.strip().split(",") for l in open("candidates.csv")]
    probs = [l.strip().split(",") for l in open("test_subsumption_scores.csv")]
    return {f"{r[0][len(NS):]} ⊑ {c[len(NS):]}".lower(): float(p)
            for r, ps in zip(rows, probs) for c, p in zip(r[1:], ps)}

after = read_scores()
pd.Series(after, name="P(C ⊑ D)").round(2)

aunt ⊑ female     1.0
aunt ⊑ male       0.0
niece ⊑ female    1.0
niece ⊑ male      0.0
Name: P(C ⊑ D), dtype: float64

## Extra · Before vs after training

In [9]:
config.fine_tune.do_fine_tune = False          # keep BERT as downloaded
BERTSubsIntraPipeline(onto=onto, config=config)
before = read_scores()
pd.DataFrame({"before": before, "after": after}).round(2)

31 named classes, 1.0 labels per class
Positive train/valid subsumptions: 42/42


pretrained or fine-tuned model loaded.
Fine-tuning costs 0.1 minutes
Predicted subsumption scores are saved to test_subsumption_scores.csv


,before,after
aunt ⊑ female,0.58,1.0
aunt ⊑ male,0.55,0.0
niece ⊑ female,0.71,1.0
niece ⊑ male,0.62,0.0


## Extra · More you can do

In [10]:
pipeline.score([["queen", "female"], ["king", "female"]]).round(2)     # any two names

array([1., 0.])

```python
config.prompt.prompt_type = "path"          # add nearby classes: "traversal" or "path"
config.subsumption_type = "restriction"     # parents like ∃hasParent.Male
config.fine_tune.pretrained = "dmis-lab/biobert-v1.1"   # any BERT checkpoint
config.test_type = "evaluation"             # ranking scores (MRR, Hits@k)
```

**Notes.** On such a small ontology the probabilities change from run to run. Your own ontology: change `config.onto_file` and the candidates file.
Paper: Chen et al. *Contextual Semantic Embeddings for Ontology Subsumption Prediction*. World Wide Web Journal 2023. DeepOnto: krr-oxford.github.io/DeepOnto